# Schema Quality: nullable types & reasoning prompts

Schema design, not just the model, decides how good your Retab extractions are.
This notebook proves it with a small, reproducible experiment and ends with a
linter that catches the problems automatically.

**What you'll see**

1. A corpus of five invoices of the *same* type that differ only in which
   optional fields are printed.
2. Three schemas for that one corpus: the **baseline** produced by
   `retab schemas generate`, then **nullable** and **reasoning** upgrades.
3. A measured comparison (accuracy + consensus stability).
4. A static **linter** whose warnings line up with the measured failures.


## 1. The corpus

All five documents are invoices processed by one schema. They differ only in
*form* — which optional fields appear:

| Field | full | no_name | no_code | minimal | mixed |
|---|:--:|:--:|:--:|:--:|:--:|
| customer_name | ✓ | · | ✓ | · | ✓ |
| customer_code | ✓ | ✓ | · | · | ✓ |
| purchase_order | ✓ | ✓ | · | · | · |
| discount | ✓ | ✓ | ✓ | · | · |
| due_date | ✓ | ✓ | ✓ | · | ✓ |
| tax | ✓ | ✓ | ✓ | · | ✓ |

(✓ present, · absent). This is the shape that tests **nullable types**: a field
present on some invoices and genuinely absent on others. `discount` is printed
negative and several dates use European `DD/MM/YYYY`, so *convention* is tested
too. `documents/manifest.json` holds the correct value of every field.


In [1]:
# Build the corpus PDFs (only if missing) and the schema variants.
import os, sys, subprocess
if not os.path.exists("documents/invoice_full.pdf"):
    subprocess.run([sys.executable, "generate_documents.py"], check=True)
from harness.variants import build_all
build_all()  # writes schemas/nullable.json and schemas/reasoning.json from baseline.json
print("corpus and schemas ready")

wrote C:\Users\DELL\Desktop\retab\cookbook\schema-quality\schemas\nullable.json
wrote C:\Users\DELL\Desktop\retab\cookbook\schema-quality\schemas\reasoning.json
corpus and schemas ready


## 2. The three schemas

Each variant adds exactly one lever, so every difference is attributable.

- **baseline** — `retab schemas generate` from one invoice. Every field
  `required`, no nullable types, no reasoning prompts.
- **nullable** — the optional fields retyped `["<type>", "null"]`. They stay
  `required`: under strict structured output every property is emitted anyway,
  so optionality is carried by the null type, not by dropping the key.
- **reasoning** — nullable plus an **`X-ReasoningPrompt`** per optional field.

**What is a reasoning prompt?** `X-ReasoningPrompt` is a Retab schema extension.
Its text is given to the model as a private, per-field "think about it this way"
instruction; it never appears in the output, it only steers extraction. Here it
says, per field, *"only return this if it is actually printed; if absent return
null; the discount is negative."* `X-SystemPrompt` is the same idea for the
whole schema.

Look at one field, `discount_amount`, across the three schemas:

In [2]:
import json
def show(field):
    for name in ["baseline", "nullable", "reasoning"]:
        prop = json.load(open(f"schemas/{name}.json", encoding="utf-8"))["properties"][field]
        print(f"--- {name} ---")
        print("  type:", prop.get("type"))
        if "X-ReasoningPrompt" in prop:
            print("  X-ReasoningPrompt:", prop["X-ReasoningPrompt"][:80] + "...")
show("discount_amount")

--- baseline ---
  type: number
--- nullable ---
  type: ['number', 'null']
--- reasoning ---
  type: ['number', 'null']
  X-ReasoningPrompt: Return the discount only if a discount line is present (use the printed value, t...


## 3. Run the experiment

For each (invoice, variant) we extract at `n_consensus=5` via the Retab CLI,
score against the manifest, and report. Results are cached under `results/`, so
re-running this cell is free.

In [ ]:
from harness.experiment import run
from IPython.display import Markdown, display
display(Markdown(run(n_consensus=5, model="retab-micro")))

## 4. Interpretation

Three findings — one per lever, including an honest null result:

- **Absent optional numbers → the baseline invents `0`.** With no discount (and
  no tax on `minimal`), the required non-nullable number had no way to say
  "absent", so it returned `0`. Nullability fixed it (`null`): absent-field
  accuracy jumped **73% → 100%**. *Mark genuinely-optional fields nullable.*
  (The baseline returned `null` for absent *strings* on its own — the reliable
  failure was on numbers, where `0` looks legitimate.)
- **Present field, wrong convention → reasoning helps, but only partly here.**
  The discount is printed negative, yet baseline and nullable returned the wrong
  sign (`150` instead of `-150`) on every invoice that had one. The reasoning
  prompt flipped it on `invoice_no_name`, dropped confidence from `1.00` to
  `0.60` on `invoice_no_code` (a useful "I'm unsure" flag), but still returned
  `150` on `invoice_full`. Present-field accuracy rose **94% → 96%**. *Reasoning
  prompts encode conventions a bare type can't — but on a small model they
  mitigate, they don't guarantee.*
- **Null result: dates needed no help.** European `DD/MM/YYYY` dates were read
  correctly by `retab-micro` with no prompt. *Not every risky-looking pattern
  actually breaks — measure before adding complexity.*

Consensus likelihood stayed high (~0.99–1.00) and was a confident `1.00` even
when the baseline had the sign **wrong** — high confidence does not imply
correctness. The one place reasoning pulled it down, to `0.60`, was itself a
signal worth heeding.

## 5. The linter

The experiment cost credits to run once. The linter turns its lessons into a
free, instant static check: it reads a schema and flags the same patterns —
no API calls, no documents.


In [4]:
from harness.lint import lint_schema
import json
for name in ["baseline", "nullable", "reasoning"]:
    schema = json.load(open(f"schemas/{name}.json", encoding="utf-8"))
    findings = lint_schema(schema)
    warns = [f for f in findings if f.severity == "warn"]
    print(f"{name:9s}  {len(warns)} warn")
    for f in warns:
        print(f"    [WARN] {f.path}: {f.rule}")

baseline   2 warn
    [WARN] <schema>: all-required-no-nullable
    [WARN] discount_amount: sign-convention-no-reason
nullable   1 warn
    [WARN] discount_amount: sign-convention-no-reason
reasoning  0 warn


### The payoff

The linter's `warn` count tracks the measured accuracy — more warnings, more failures:

| Schema | Linter warns | Measured accuracy |
|---|:--:|:--:|
| baseline | 2 (all-required + discount-sign) | 90% |
| nullable | 1 (discount-sign still) | 95% |
| reasoning | 0 (clean) | 97% |

The static checks move in lock-step with the live measurements, so the linter
encodes proven patterns rather than guesses. (The remaining 3% is the discount
*sign* — a convention the prompt only partly fixes on `retab-micro`; the linter
flags it as the `discount-sign` warning the reasoning schema clears.)

**Takeaways**

1. Mark genuinely-optional fields **nullable** — a required, non-nullable field fabricates values (numbers become `0`). Keep the field `required` but nullable: that's the right shape for strict structured output.
2. Use an **`X-ReasoningPrompt`** to encode conventions a type can't (signs, units, tie-breakers) — and verify it actually lands on your model.
3. **Measure** — confidence ≠ correctness, and not every risky pattern actually breaks.
4. Run `python -m harness.lint your_schema.json` before you ship a schema.